# Lab 19: Tree-Based Models — Random Forests
## ECON 3916: Data Science for Economists
### Guided Construction Lab | 30 min Core + 15 min Extension

---

**Learning Objectives:**
- Compare a single decision tree, Ridge regression, and Random Forest on the same dataset
- Tune Random Forest hyperparameters via cross-validation
- Extract and interpret feature importance (MDI and permutation)
- Apply the Ch 18 evaluation toolkit (confusion matrix, AUC) to a RF classifier

**Dataset:** California Housing (sklearn) — recurring thread from Ch 4, 12-16

**Foundations First Policy:** Part 1-2 are GUIDED (run as-is, interpret results). Part 3 has YOUR TASK sections (fill in the blanks). Extension is open-ended.

---

## Setup

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load California Housing data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score, 
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load California Housing (you've seen this dataset since Ch 4)
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set: {X_train.shape[0]} observations, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} observations')
print(f'Features: {list(X.columns)}')

## Part 1: Decision Tree vs. Random Forest vs. Ridge (GUIDED — 10 min)

We compare three models on the same data. The question: does the Random Forest's
added complexity produce meaningfully better predictions than simpler alternatives?

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 2: Train and compare Decision Tree, Ridge, and RF
# -----------------------------------------------------------

# 1. Unrestricted Decision Tree (high variance, low bias)
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

# 2. Ridge Regression (Ch 16 callback — stable but linear)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# 3. Random Forest (100 trees, default settings)
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

# Compare all three
results = []
for name, model in [('Single Tree', tree), ('Ridge (Ch 16)', ridge), ('Random Forest', rf)]:
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Train RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train R\u00b2': r2_score(y_train, train_pred),
        'Test R\u00b2': r2_score(y_test, test_pred),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4))

### Interpretation Questions (answer in a markdown cell below)

1. Which model has the largest gap between Train R² and Test R²? What does this tell you about its variance?
2. The Random Forest's Test R² is substantially higher than Ridge. What does this suggest about the relationship between features and housing prices — is it linear or non-linear?
3. **Callback to Ch 15:** Where does each model sit on the bias-variance U-curve?

*Your answers here:*

1. The Single Decision Tree has the largest Train/Test R² gap. An unrestricted tree perfectly memorizes training data (Train R² ≈ 1.0) but generalizes poorly. The gap is its bias-variance fingerprint.
2. the relationship between features and housing prices is non-linear. Ridge is a linear model and hits a ceiling regardless of regularization strength. The RF's superior test performance means it is capturing interaction effects and non-linear feature relationships.
3. Single Tree — far right: low bias, very high variance (overfit)
Ridge — left-of-center: high bias (linearity constraint), low variance
Random Forest — near the trough: moderate bias, variance substantially reduced by bagging, sitting closest to the optimal point on the curve

## Part 2: Feature Importance — MDI vs. Permutation (GUIDED — 10 min)

Random Forests tell us which features matter most for prediction.
But remember: **predictive importance ≠ causal importance.**

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 3: Compare MDI and permutation feature importance
# -----------------------------------------------------------
mdi_importance = pd.Series(
    rf.feature_importances_, index=X.columns
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MDI importance
mdi_importance.plot.barh(ax=axes[0], color='#6B8BA4')
axes[0].set_title('MDI Feature Importance (from training)', fontsize=12)
axes[0].set_xlabel('Importance')

# Permutation importance (more reliable — uses test set)
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE
)
perm_importance = pd.Series(
    perm_result.importances_mean, index=X.columns
).sort_values(ascending=True)

perm_importance.plot.barh(ax=axes[1], color='#C47B5A')
axes[1].set_title('Permutation Importance (from test set)', fontsize=12)
axes[1].set_xlabel('Importance (decrease in R\u00b2)')

plt.tight_layout()
plt.savefig('figures/ch19_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 by MDI:', mdi_importance.tail(3).index.tolist())
print('Top 3 by Permutation:', perm_importance.tail(3).index.tolist())

### Interpretation Questions

1. Do MDI and permutation importance agree on the top features? Where do they diverge?
2. Why might MDI overstate the importance of `Latitude` and `Longitude` (hint: how many unique values do they have)?
3. **Critical question:** If `MedInc` is the top feature, does that mean raising median income in a neighborhood will raise housing prices? Why or why not? (Hint: Chapter 10 DAGs)

*Your answers here:*

1. Both methods agree that MedInc is the dominant predictor. They typically agree on the top 2–3 features overall. Divergence tends to appear in Latitude and Longitude, which MDI tends to overrank relative to permutation importance.
2. MDI scores are computed at split time on training data and are biased toward high-cardinality continuous features. Latitude and Longitude have near-infinite unique values, giving the tree many possible split points to exploit. Permutation importance corrects for this because it measures actual degradation on held-out test data, not training split quality.
3. No. Feature importance measures predictive association, not causal effect. From a Ch. 10 DAG perspective, MedInc and SalePrice are likely both downstream of unobserved confounders (neighborhood desirability, school quality, proximity to employment centers). Without a valid instrument or natural experiment, the RF coefficient is not identified as causal. Raising a neighborhood's median income via policy could theoretically raise prices, but the model cannot tell you that.

## Part 3: Hyperparameter Tuning & Classification (YOUR TASK — 10 min)

Now YOU tune a Random Forest and apply it to a classification problem.

In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Tune the Random Forest with GridSearchCV
# -----------------------------------------------------------

# The parameter grid is given — these are the hyperparameters to search over
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.5, None],  # sqrt = sqrt(p), 0.5 = half, None = all
}

# TODO: Create a GridSearchCV object
# Hints:
#   - estimator: RandomForestRegressor with random_state=RANDOM_STATE
#   - param_grid: use the param_grid defined above
#   - cv: 5-fold cross-validation
#   - scoring: 'neg_mean_squared_error'
#   - n_jobs: -1 (use all cores)

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)

# Fit on training data
grid_search.fit(X_train, y_train)

# TODO: Fit the grid search on the training data

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}')

# Evaluate best model on test set
best_rf = grid_search.best_estimator_
test_rmse = np.sqrt(mean_squared_error(y_test, best_rf.predict(X_test)))
test_r2 = r2_score(y_test, best_rf.predict(X_test))
print(f'Tuned RF — Test RMSE: {test_rmse:.4f}, Test R²: {test_r2:.4f}')

In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Classification: Predict expensive homes (> median)
# -----------------------------------------------------------

# Step 1: Create binary target — 1 if price > median, 0 otherwise
# Hint: use np.median(y) and .astype(int)
median_price = np.median(y)
y_binary     = (y > median_price).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_binary, test_size=0.2, random_state=RANDOM_STATE
)

# Step 2: Fit an RF classifier (200 trees) and Logistic Regression (Ch 17 callback)
# Hint: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
#       LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
rf_clf  = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_clf.fit(X_train_c, y_train_c)

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_c, y_train_c)

# Compare AUC (Ch 18 evaluation toolkit) — GUIDED, run as-is
for name, model in [('Logistic Regression (Ch 17)', log_reg), ('Random Forest', rf_clf)]:
    y_proba = model.predict_proba(X_test_c)[:, 1]
    auc = roc_auc_score(y_test_c, y_proba)
    print(f'{name:35s} — AUC: {auc:.4f}')

### Final Interpretation

1. How much did hyperparameter tuning improve the RF over default settings?
2. Does the RF classifier beat logistic regression on AUC? Is the difference practically significant?
3. If you were deploying this for a real estate company, which model would you choose and why? Consider: accuracy, interpretability, maintenance cost.

*Your answers here:*

1. Typically 2–5% reduction in Test RMSE over default settings. The default RF is already a strong baseline because bagging handles most of the variance reduction. Tuning max_depth and max_features refines the bias-variance balance at the individual tree level, but the ensemble mechanism does most of the heavy lifting regardless.
2. Yes, RF will achieve a higher AUC (typically ~0.93–0.95 vs ~0.88–0.90 for logistic regression on this dataset). Whether it's practically significant depends on the cost structure of the application. A 3–5 point AUC difference in a financial context (e.g., flagging expensive homes for targeted marketing) is operationally meaningful; in a low-stakes context it may not justify the added model complexity.
3. For most production settings, Random Forest is the right call on performance grounds

---

## Extension (Optional — 15 min): Partial Dependence Plots

Partial dependence plots show the marginal effect of a feature on the prediction,
averaging over all other features. This reveals non-linearities the RF captures.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: Partial dependence plots for top features
# -----------------------------------------------------------
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(12, 4))
PartialDependenceDisplay.from_estimator(
    best_rf, X_test, features=['MedInc', 'AveOccup'],
    kind='average', ax=ax
)
plt.suptitle('Partial Dependence \u2014 RF captures non-linear effects OLS misses', fontsize=13)
plt.tight_layout()
plt.savefig('figures/ch19_partial_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

---
## AI-Assisted Expansion: Interactive Random Forest Dashboard

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over decision trees, random forests, hyperparameter tuning, and feature importance. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task
Build an interactive Streamlit app (or Plotly dashboard with ipywidgets) that lets the user:
1. Adjust `n_estimators` (1-500) and `max_features` (1-8) with sliders
2. See the prediction surface update in real-time
3. Compare RF vs Ridge vs single tree performance as hyperparameters change
4. Display feature importance that updates with each parameter change

This transforms your static lab output into a reusable teaching artifact for your portfolio.

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# Copy the P.R.I.M.E. prompt above into Claude, then paste
# the generated code here. Run it and verify.
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in interactive visualizations and scikit-learn.
#
# [Request] I just completed a lab where I compared Decision Trees,
# Ridge regression, and Random Forests on California Housing data.
# I tuned RF hyperparameters with GridSearchCV and extracted MDI
# and permutation feature importance. Now I want to expand this
# into an interactive Plotly dashboard with ipywidgets sliders
# for n_estimators (1-500) and max_features (1-8). The dashboard
# should update three panels: (1) model comparison bar chart,
# (2) feature importance that changes with max_features,
# (3) the Train vs Test R\u00b2 as n_estimators increases.
#
# [Iterate] Use plotly.graph_objects, ipywidgets, numpy, sklearn.
# Use the same variable names: X_train, X_test, y_train, y_test,
# data.feature_names. Do not use deprecated Plotly functions.
#
# [Mechanism Check] Add inline comments explaining how ipywidgets
# observers trigger plot updates and why we re-fit inside the callback.
#
# [Evaluate] Explain what the dashboard reveals about the relationship
# between n_estimators, max_features, and test performance.

# PASTE AI-GENERATED CODE BELOW:


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# INTERACTIVE RANDOM FOREST DASHBOARD
# Lab 19 — ECON 3916: Tree-Based Models
# Requires: sklearn, plotly, ipywidgets, numpy
# Run in Jupyter Notebook or JupyterLab
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── 1. DATA SETUP ─────────────────────────────────────────────────────────────
# Same variable names as the lab — drop this block if already loaded
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training set : {X_train.shape[0]} obs × {X_train.shape[1]} features")
print(f"Test set     : {X_test.shape[0]} obs")

# ── 2. FIT STATIC BASELINES (fitted once — not sensitive to RF hyperparams) ──
# Ridge and single Decision Tree are fixed reference points on every chart

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_test_r2   = r2_score(y_test, ridge.predict(X_test))
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, ridge.predict(X_test)))

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
tree_test_r2    = r2_score(y_test, tree.predict(X_test))
tree_train_r2   = r2_score(y_train, tree.predict(X_train))
tree_test_rmse  = np.sqrt(mean_squared_error(y_test, tree.predict(X_test)))

print(f"\nBaseline — Ridge  : Test R² = {ridge_test_r2:.4f}  RMSE = {ridge_test_rmse:.4f}")
print(f"Baseline — Tree   : Test R² = {tree_test_r2:.4f}  RMSE = {tree_test_rmse:.4f}")

# ── 3. CONVERGENCE CURVE HELPER ───────────────────────────────────────────────
# Builds the n_estimators learning curve by fitting RF at increasing tree counts.
# We re-fit inside this function because ipywidgets observers fire on every
# slider change — each new (n_estimators, max_features) combo is a new model.

def build_convergence_curve(max_n, max_features, steps=None):
    """
    Fit RF at each step up to max_n trees and record Train/Test R².
    Returns lists of (n_vals, train_r2s, test_r2s).
    
    Why re-fit here instead of using warm_start?
    warm_start only adds trees — it can't change max_features mid-run.
    A full re-fit is required when max_features changes.
    """
    if steps is None:
        # Sample checkpoints — sparse at high n where gains are marginal
        steps = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 400, 500]
    steps = [s for s in steps if s <= max_n]
    if not steps or steps[-1] != max_n:
        steps.append(max_n)

    n_vals, train_r2s, test_r2s = [], [], []

    for n in steps:
        rf_curve = RandomForestRegressor(
            n_estimators=n,
            max_features=max_features,
            random_state=RANDOM_STATE,
            n_jobs=-1          # parallelise across cores
        )
        rf_curve.fit(X_train, y_train)
        train_r2s.append(r2_score(y_train, rf_curve.predict(X_train)))
        test_r2s.append(r2_score(y_test,  rf_curve.predict(X_test)))
        n_vals.append(n)

    return n_vals, train_r2s, test_r2s


# ── 4. MAIN DASHBOARD RENDER ──────────────────────────────────────────────────
# This function is the ipywidgets observer callback.
# It fires every time either slider value changes, receiving the new values
# directly as keyword arguments via widgets.interact().

def render_dashboard(n_estimators, max_features):
    """
    Observer callback — called by ipywidgets on every slider event.
    Re-fits RF with current hyperparams, then redraws all three panels.
    """

    # ── Fit the RF with current slider values ──────────────────────────────
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    rf_train_r2   = r2_score(y_train, rf.predict(X_train))
    rf_test_r2    = r2_score(y_test,  rf.predict(X_test))
    rf_test_rmse  = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))

    # MDI feature importance — extracted directly from the fitted RF object
    # rf.feature_importances_ is a (n_features,) array of mean impurity decrease
    importance = pd.Series(
        rf.feature_importances_, index=data.feature_names
    ).sort_values(ascending=True)   # ascending=True → largest bar at top of barh

    # ── Build convergence curve data for Panel 3 ───────────────────────────
    # This is the expensive step — reduce steps for responsiveness
    n_vals, train_r2s, test_r2s = build_convergence_curve(
        max_n=n_estimators,
        max_features=max_features,
        steps=[1, 5, 10, 20, 50, 100, 200, 300, 500]
    )

    # ── Create 3-panel figure ──────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Panel 1 · Model Comparison (Test R²)",
            "Panel 2 · Feature Importance (MDI)",
            "Panel 3 · Train vs Test R² Convergence Curve"
        ),
        specs=[[{}, {}], [{"colspan": 2}, None]],
        vertical_spacing=0.18,
        horizontal_spacing=0.12
    )

    # ── PANEL 1: Model comparison bar chart ───────────────────────────────
    # Compares RF (current settings) vs Ridge vs single tree on Test R²
    # Also shows RF Train R² so the generalisation gap is visible
    models   = ["Ridge (baseline)", "Decision Tree", "RF Train R²", "RF Test R²"]
    r2_vals  = [ridge_test_r2, tree_test_r2, rf_train_r2, rf_test_r2]
    colors   = ["#7cfc8a", "#ff4d6d", "rgba(0,180,216,0.4)", "#00b4d8"]

    fig.add_trace(
        go.Bar(
            x=r2_vals,
            y=models,
            orientation='h',
            marker_color=colors,
            text=[f"{v:.4f}" for v in r2_vals],
            textposition='outside',
            cliponaxis=False,
            name="R²"
        ),
        row=1, col=1
    )
    fig.update_xaxes(range=[0, 1.05], title_text="R²", row=1, col=1)

    # ── PANEL 2: Feature importance (MDI) ─────────────────────────────────
    # max_features controls how many candidates each split considers.
    # Lower max_features → stronger MedInc dominance (it wins the random draw
    # whenever it appears). Higher → more even spread as weaker features compete.
    fig.add_trace(
        go.Bar(
            x=importance.values,
            y=importance.index,
            orientation='h',
            marker=dict(
                color=importance.values,
                colorscale=[[0, "#1a2235"], [1, "#ff6b35"]],
                showscale=False
            ),
            text=[f"{v:.3f}" for v in importance.values],
            textposition='outside',
            cliponaxis=False,
            name="MDI Importance"
        ),
        row=1, col=2
    )
    fig.update_xaxes(title_text="Mean Impurity Decrease", row=1, col=2)

    # ── PANEL 3: Convergence curve ─────────────────────────────────────────
    # Shows Train R² and Test R² as n_estimators grows from 1 → current value.
    # Key insight: test R² converges (diminishing returns) while train R²
    # stays high — the gap is the generalisation error floor.
    fig.add_trace(
        go.Scatter(
            x=n_vals, y=train_r2s,
            mode='lines+markers',
            name='Train R²',
            line=dict(color='rgba(0,180,216,0.4)', width=2, dash='dot'),
            marker=dict(size=5)
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=n_vals, y=test_r2s,
            mode='lines+markers',
            name='Test R²',
            line=dict(color='#00b4d8', width=3),
            marker=dict(size=7)
        ),
        row=2, col=1
    )

    # Reference lines for Ridge and Tree on Panel 3
    fig.add_hline(
        y=ridge_test_r2, row=2, col=1,
        line=dict(color='#7cfc8a', width=1, dash='dash'),
        annotation_text=f"Ridge R²={ridge_test_r2:.3f}",
        annotation_position="bottom right"
    )
    fig.add_hline(
        y=tree_test_r2, row=2, col=1,
        line=dict(color='#ff4d6d', width=1, dash='dash'),
        annotation_text=f"Tree R²={tree_test_r2:.3f}",
        annotation_position="top right"
    )

    fig.update_xaxes(title_text="n_estimators", row=2, col=1)
    fig.update_yaxes(title_text="R²", range=[0.3, 1.02], row=2, col=1)

    # ── Layout ─────────────────────────────────────────────────────────────
    fig.update_layout(
        title=dict(
            text=(
                f"<b>Random Forest Dashboard</b> — "
                f"n_estimators={n_estimators} · max_features={max_features}<br>"
                f"<sup>RF Test R²={rf_test_r2:.4f} · RMSE={rf_test_rmse:.4f} · "
                f"Ridge R²={ridge_test_r2:.4f} · Tree R²={tree_test_r2:.4f}</sup>"
            ),
            font=dict(size=15)
        ),
        height=700,
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
        paper_bgcolor='#0a0e17',
        plot_bgcolor='#111827',
        font=dict(color='#e2e8f0', family='monospace'),
        margin=dict(t=120, b=40, l=60, r=40)
    )

    # Style all axes consistently
    axis_style = dict(
        gridcolor='#1e2d45',
        linecolor='#1e2d45',
        tickfont=dict(size=10)
    )
    fig.update_xaxes(**axis_style)
    fig.update_yaxes(**axis_style)

    fig.show()


# ── 5. WIDGET DEFINITION & OBSERVER WIRING ────────────────────────────────────
# ipywidgets.interact() wires each slider to the render_dashboard callback.
# When a slider fires an 'observe' event (value changes), interact() calls
# render_dashboard(n_estimators=<new_val>, max_features=<new_val>) automatically.
# clear_output(wait=True) prevents figures from stacking up in the notebook.

n_est_slider = widgets.IntSlider(
    value=100, min=1, max=500, step=1,
    description='n_estimators',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)

max_feat_slider = widgets.IntSlider(
    value=3, min=1, max=8, step=1,
    description='max_features',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)

# Use interact_manual to avoid re-fitting on every pixel drag (expensive)
# Change to widgets.interact() if you want live updates on every tick
out = widgets.interact_manual(
    render_dashboard,
    n_estimators=n_est_slider,
    max_features=max_feat_slider
)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x11022d790>>
Traceback (most recent call last):
  File "/Users/mayaellis/anaconda3/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Training set : 16512 obs × 8 features
Test set     : 4128 obs

Baseline — Ridge  : Test R² = 0.5759  RMSE = 0.7455
Baseline — Tree   : Test R² = 0.6187  RMSE = 0.7069


interactive(children=(IntSlider(value=100, description='n_estimators', layout=Layout(width='500px'), max=500, …

---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Compared Decision Tree, Ridge Regression, and Random Forest on
#   California Housing data (20,640 observations, 8 features)
# * Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth,
#   max_features)
# * Extracted and compared MDI vs permutation feature importance
# * Built an RF classifier and compared AUC against logistic regression
# * Created an interactive dashboard with Plotly + ipywidgets
# * Key finding: RF achieved R\u00b2 = [YOUR VALUE] vs Ridge R\u00b2 = [YOUR VALUE]
#
# **Please write a README.md entry including:**
# 1. Project Title: Tree-Based Models \u2014 Random Forests
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.